# Supplementary Results 1 — Systematic ancestry-specific fine-mapping catalogue

Every number claimed in Supplementary Results section 1. Numbers already checked as part of
Results 1 (100,526 studies, 789,453 credible sets, 39,282 studies with a credible set, 4,250
publications, 9,280 ontology terms, 2,044,305 molQTL credible sets, 98 tissues, 520,975 /
70,618 / 450,357 qualifying credible sets, 263,705 and 1,461,445 replicated credible sets) are
recomputed here for context but registered only once, in `results/panoramic.json`.

Numbers are collected in `numbers` and written to `results/sr01_finemapping_catalogue.json`,
which `tools/check_numbers.py` compares against the manuscript.

**Provenance.** Most of this section was computed in
`chapters/_legacy/02-analysis/01-descriptions-numbers/01_descriptive_numbers.ipynb` (and its
original, `~/Projects/EGL_and_training_set/archive/gentropy_paper/02_descriptive_numbers_si_vi_fm.ipynb`),
whose `stats_from_list()` produced the per-source statistics of Supplementary Table 10. Two
claims have no surviving code and are reimplemented from the prose, flagged below: the credible
set size versus MAF regression, and the Cochran heterogeneity of lead variant-disease pairs.

In [1]:
import numpy as np
import pandas as pd
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from pyspark.sql import functions as f
from scipy import stats

from manuscript_methods import discovery, paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
numbers = {}

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/20 12:59:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Precomputed numbers

Two numbers in this section were produced **upstream of this pipeline**, at ingestion time and
before the validation this repository starts from. They are recorded here rather than recomputed,
so that anyone reading the notebook can see what they are and why nothing below produces them.
`tools/check_numbers.py` reports them as PRECOMPUTED rather than as failures.

In [2]:
PRECOMPUTED = pd.DataFrame(
    [
        {
            "id": "S1.02",
            "claim": "27.3% of all studies had at least one credible set",
            "value": 27.3,
            "why it is not recomputed": "the denominator is the GWAS Catalog study set before Open "
            "Targets ingested it; the release holds only what was ingested",
            "input that would close it": "upstream .../study_index/gwas_catalog",
        },
        {
            "id": "S1.28",
            "claim": "around 30% of GWAS Catalog studies excluded before fine-mapping",
            "value": 30.0,
            "why it is not recomputed": "the excluded studies are absent from the release; its "
            "qualityControls field records post-ingestion reasons only",
            "input that would close it": "upstream .../study_index/gwas_catalog",
        },
    ]
)
PRECOMPUTED

,id,claim,value,why it is not recomputed,input that would close it
0,S1.02,27.3% of all studies had at least one credible...,27.3,the denominator is the GWAS Catalog study set ...,upstream .../study_index/gwas_catalog
1,S1.28,around 30% of GWAS Catalog studies excluded be...,30.0,the excluded studies are absent from the relea...,upstream .../study_index/gwas_catalog


## Studies, credible sets and the share of studies that fine-mapped

`stats_from_list()` counted binary traits as studies with `nCases > 0` among the studies that
produced at least one credible set.

In [3]:
si = StudyIndex.from_parquet(session, paper.release("study")).df.cache()
cs = StudyLocus.from_parquet(session, paper.release("credible_set")).df.cache()

studies_with_cs_ids = cs.select("studyId").distinct().cache()
n_studies = si.count()
n_with_cs = studies_with_cs_ids.count()

gwas = si.filter(f.col("studyType") == "gwas").cache()
gwas_cs = cs.filter(f.col("studyType") == "gwas").cache()
gwas_with_cs = gwas.join(studies_with_cs_ids, "studyId", "inner").cache()

print(f"studies in the release index: {n_studies:,}")
print(f"studies with at least one credible set: {n_with_cs:,}")
print(f"GWAS studies: {gwas.count():,} | GWAS credible sets: {gwas_cs.count():,}")
print(f"GWAS studies with at least one credible set: {gwas_with_cs.count():,}")

26/08/20 12:59:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


studies in the release index: 1,966,178
studies with at least one credible set: 1,864,151


GWAS studies: 100,526 | GWAS credible sets: 789,453


GWAS studies with at least one credible set: 39,282


In [4]:
# 27.3% is a **precomputed** number: it counts the GWAS Catalog studies that entered fine-mapping,
# before the validation this repository starts from, and no denominator in the release gives it.
# 39,282 / 0.273 = 143,890, about 43% more GWAS studies than the release ingested, so the
# denominator is the GWAS Catalog set *before* ingestion — the same quantity Supplementary Table 10
# calls "Original number of studies before ingestion". Candidates are printed for the record only.
candidates = {
    "all studies in the index": (n_with_cs, n_studies),
    "GWAS studies only": (gwas_with_cs.count(), gwas.count()),
    "molQTL studies only": (n_with_cs - gwas_with_cs.count(), n_studies - gwas.count()),
}
for name, (num, den) in candidates.items():
    print(f"{name}: {num:,} / {den:,} = {100 * num / den:.1f}%")
print(f"denominator implied by the published 27.3%: {39282 / 0.273:,.0f} studies")

all studies in the index: 1,864,151 / 1,966,178 = 94.8%
GWAS studies only: 39,282 / 100,526 = 39.1%
molQTL studies only: 1,824,869 / 1,865,652 = 97.8%
denominator implied by the published 27.3%: 143,890 studies


In [5]:
n_binary = gwas_with_cs.filter(f.col("nCases").isNotNull() & (f.col("nCases") > 0)).count()
numbers["S1.01"] = round(100 * n_binary / gwas_with_cs.count(), 2)

earliest = (
    gwas_with_cs.filter(f.col("publicationDate").isNotNull())
    .select(f.year(f.to_date("publicationDate", "yyyy-MM-dd")).alias("year"))
    .agg(f.min("year"))
    .collect()[0][0]
)
numbers["S1.11"] = int(earliest)
print(f"binary traits among GWAS with a credible set: {numbers['S1.01']}%")
print(f"earliest publication year: {numbers['S1.11']}")

binary traits among GWAS with a credible set: 20.24%
earliest publication year: 2006


## Ancestry composition of the GWAS that fine-mapped

`discovery.classify_ancestry` is the classification used for Figure 1c: EUR when non-Finnish
Europeans reach 90% of the LD population structure, non-EUR when another single ancestry does,
mixed when none does.

In [6]:
classified = discovery.classify_ancestry(gwas_with_cs).cache()
total = classified.count()
counts = classified.groupBy("ancestryClass").count().toPandas().set_index("ancestryClass")["count"]

numbers["S1.04"] = round(100 * counts.get("non-EUR", 0) / total, 1)
numbers["S1.05"] = round(100 * counts.get("mixed", 0) / total, 1)
numbers["S1.03"] = round(numbers["S1.04"] + numbers["S1.05"], 1)
print(counts.to_string())
print(f"not predominantly NFE: {numbers['S1.03']}% (non-EUR {numbers['S1.04']}%, mixed {numbers['S1.05']}%)")

ancestryClass
mixed       5393
EUR        26736
non-EUR     7153
not predominantly NFE: 31.9% (non-EUR 18.2%, mixed 13.7%)


### Which ancestry predominates within each class

The published breakdown is over **all** GWAS studies, not only those that produced a credible set:
the same classification feeds the Figure 1d ancestry donut, whose AFR and EAS/CSA wedges (7,421 and
11,785 studies) stand in the published 31.5 : 50.0 ratio exactly. Both denominators are computed;
the all-study one is registered.

In [7]:
classified_all = discovery.classify_ancestry(gwas).cache()


def ancestry_breakdown(frame):
    """Percentage of non-EUR studies by predominant ancestry, and the NFE share of mixed ones."""
    non_eur = (
        frame.filter(f.col("ancestryClass") == "non-EUR")
        .groupBy("predominantAncestry")
        .count()
        .toPandas()
        .set_index("predominantAncestry")["count"]
    )
    mixed = frame.filter(f.col("ancestryClass") == "mixed")
    nfe_share = round(100 * mixed.filter(f.col("predominantAncestry") == "nfe").count() / mixed.count(), 1)
    return (100 * non_eur / non_eur.sum()).round(1), nfe_share


share_all, mixed_nfe_all = ancestry_breakdown(classified_all)
share_cs, mixed_nfe_cs = ancestry_breakdown(classified)
comparison = pd.DataFrame({"all GWAS studies": share_all, "GWAS with a credible set": share_cs})
print(comparison.sort_values("all GWAS studies", ascending=False).to_string())
print(f"mixed studies whose largest ancestry is NFE: {mixed_nfe_all}% (all), {mixed_nfe_cs}% (with a CS)")

numbers["S1.06"] = float(share_all.get("eas", np.nan))
numbers["S1.07"] = float(share_all.get("afr", np.nan))
numbers["S1.08"] = float(share_all.get("fin", np.nan))
numbers["S1.09"] = float(share_all.get("amr", np.nan))
numbers["S1.10"] = mixed_nfe_all

                     all GWAS studies  GWAS with a credible set
predominantAncestry                                            
eas                              50.0                      40.2
afr                              31.5                      34.5
fin                               9.8                      17.4
amr                               8.7                       7.9
mixed studies whose largest ancestry is NFE: 82.7% (all), 95.3% (with a CS)


## Credible set size, and single-variant resolution by data source

The prose defines single-variant resolution as a credible set holding one variant with
PIP > 0.9. The published percentages (13.5% FinnGen, 40.1% GWAS Catalog summary statistics) come
from the looser rule used in Supplementary Table 10 — the credible set *contains* a variant with
PIP > 0.9 — so that is the rule registered here. Both are printed.

In [8]:
sizes = gwas_cs.select(
    "studyId",
    f.size("locus").alias("csSize"),
    f.array_max(f.transform("locus", lambda x: x["posteriorProbability"])).alias("maxPip"),
).cache()

summary = sizes.agg(
    f.mean("csSize").alias("mean"),
    f.expr("percentile_approx(csSize, 0.5)").alias("median"),
).collect()[0]
numbers["S1.12"] = round(float(summary["mean"]), 2)
numbers["S1.13"] = int(summary["median"])
print(f"GWAS credible set size: mean {numbers['S1.12']}, median {numbers['S1.13']}")

GWAS credible set size: mean 24.61, median 5


In [9]:
# Same source predicates as chapters/06-supplementary-tables/04_fine_mapping_numbers.ipynb.
SOURCES = {
    "GWAS Catalog Curated Associations": (f.col("projectId") == "GCST")
    & (~f.coalesce(f.col("hasSumstats"), f.lit(False))),
    "GWAS Catalog Summary Statistics": (f.col("projectId") == "GCST") & f.coalesce(f.col("hasSumstats"), f.lit(False)),
    "FinnGen R12": f.col("projectId") == "FINNGEN_R12",
}

rows = []
for name, predicate in SOURCES.items():
    subset = sizes.join(gwas.filter(predicate).select("studyId"), "studyId", "inner").cache()
    n = subset.count()
    rows.append(
        {
            "source": name,
            "credible sets": n,
            "contains PIP > 0.9 (%)": round(100 * subset.filter(f.col("maxPip") > 0.9).count() / n, 1),
            "single variant with PIP > 0.9 (%)": round(
                100 * subset.filter((f.col("csSize") == 1) & (f.col("maxPip") > 0.9)).count() / n, 1
            ),
        }
    )
resolution = pd.DataFrame(rows).set_index("source")
numbers["S1.14"] = float(resolution.loc["FinnGen R12", "contains PIP > 0.9 (%)"])
numbers["S1.15"] = float(resolution.loc["GWAS Catalog Summary Statistics", "contains PIP > 0.9 (%)"])
resolution

,credible sets,contains PIP > 0.9 (%),single variant with PIP > 0.9 (%)
source,,,
GWAS Catalog Curated Associations,153568,20.5,18.0
GWAS Catalog Summary Statistics,615181,40.1,37.6
FinnGen R12,20704,13.5,10.9


## Credible set size against MAF and against sample size

No surviving code produced the published slope of 14.0, and the sentence does not say which
credible sets it covers — over every GWAS credible set the slope is 45.2, and plausible subsets
span 3.6 to 72.8. **The definition adopted here is the qualifying credible sets**, the analysis set
used everywhere else in this work. Both regressions of this paragraph use it, and the alternatives
are exported beside them.

In [10]:
lve = session.spark.read.parquet(paper.derived("lead_variant_effect"))
qualifying_cs = (
    session.spark.read.parquet(paper.derived("qualifying_credible_sets"))
    .select("studyLocusId")
    .union(session.spark.read.parquet(paper.derived("qualifying_measurement_credible_sets")).select("studyLocusId"))
    .distinct()
    .cache()
)

regression_input = (
    lve.filter(f.col("studyStatistics.studyType") == "gwas")
    .select(
        "studyLocusId",
        f.col("locusStatistics.locusSize").alias("csSize"),
        f.col("majorLdPopulationMaf.value").alias("maf"),
        f.col("studyStatistics.nSamples").alias("nSamples"),
        f.col("studyStatistics.nCases").alias("nCases"),
        f.col("studyStatistics.nControls").alias("nControls"),
        f.col("finemappingMethod").alias("method"),
    )
    .join(qualifying_cs.withColumn("qualifying", f.lit(True)), "studyLocusId", "left")
    .filter(f.col("maf").isNotNull() & f.col("csSize").isNotNull())
    .toPandas()
)
regression_input["qualifying"] = regression_input["qualifying"].fillna(False)
print(f"GWAS credible sets with a lead variant MAF: {len(regression_input):,}")

GWAS credible sets with a lead variant MAF: 787,112


/var/folders/p5/4t9crp1563l792qz8xz_3x5h0000gq/T/ipykernel_12583/2806328494.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [11]:
def size_regression(frame, x, label):
    """Least-squares fit of credible set size on one predictor."""
    sub = frame.dropna(subset=[x, "csSize"])
    fit = stats.linregress(sub[x], sub["csSize"])
    return {"subset": label, "n": len(sub), "slope": round(fit.slope, 3), "P": fit.pvalue, "r": round(fit.rvalue, 4)}


maf_variants = pd.DataFrame(
    [
        size_regression(regression_input, "maf", "all GWAS credible sets"),
        size_regression(regression_input[regression_input["qualifying"]], "maf", "qualifying credible sets"),
        size_regression(regression_input[regression_input["maf"] >= 0.01], "maf", "common lead variants"),
        size_regression(
            regression_input[regression_input["method"].str.contains("susie", case=False, na=False)],
            "maf",
            "SuSiE only",
        ),
        size_regression(regression_input[regression_input["csSize"] <= 100], "maf", "credible sets of at most 100"),
    ]
)
maf_variants.to_csv(paper.derived("sr1_cs_size_regressions.csv"), index=False)
numbers["S1.16"] = float(maf_variants[maf_variants["subset"] == "qualifying credible sets"].iloc[0]["slope"])
print(f"adopted definition: qualifying credible sets, slope {numbers['S1.16']}")
maf_variants

adopted definition: qualifying credible sets, slope 25.969


,subset,n,slope,P,r
0,all GWAS credible sets,787112,45.208,0.000000e+00,0.0696
1,qualifying credible sets,520975,25.969,3.120209e-119,0.0322
2,common lead variants,679900,24.686,3.266822e-159,0.0326
3,SuSiE only,488979,28.841,2.202617e-215,0.0448
4,credible sets of at most 100,750811,30.976,0.000000e+00,0.2521


In [12]:
# The published sentence gives only a direction and a P value for the sample-size term. The sign
# depends on how sample size enters: over every GWAS credible set it is positive, over the
# qualifying ones it is negative on both log scales.
def sample_size_regression(frame, column, values, label, subset):
    """Credible set size against one form of study sample size."""
    fit = stats.linregress(values, frame["csSize"])
    rho, rho_p = stats.spearmanr(values, frame["csSize"])
    return {
        "subset": subset,
        "predictor": column,
        "n": len(frame),
        "slope": round(float(fit.slope), 4),
        "P": float(fit.pvalue),
        "spearman": round(float(rho), 4),
        "spearman P": float(rho_p),
    }


rows = []
for subset, frame in [
    ("all GWAS credible sets", regression_input),
    ("qualifying credible sets", regression_input[regression_input["qualifying"]]),
]:
    frame = frame.dropna(subset=["nSamples"]).copy()
    binary = frame["nCases"].notna() & frame["nControls"].notna() & (frame["nCases"] > 0)
    prevalence = (frame["nCases"] / frame["nSamples"]).where(binary)
    frame["effectiveSampleSize"] = np.where(
        binary, prevalence * (1 - prevalence) * frame["nSamples"], frame["nSamples"]
    )
    rows += [
        sample_size_regression(frame, "sample size", frame["nSamples"], "n", subset),
        sample_size_regression(frame, "log10 sample size", np.log10(frame["nSamples"].clip(lower=1)), "n", subset),
        sample_size_regression(
            frame, "log10 effective sample size", np.log10(frame["effectiveSampleSize"].clip(lower=1)), "n", subset
        ),
    ]
sample_size = pd.DataFrame(rows)
sample_size.to_csv(paper.derived("sr1_sample_size_regressions.csv"), index=False)
adopted = sample_size[
    (sample_size["subset"] == "qualifying credible sets") & (sample_size["predictor"] == "log10 effective sample size")
].iloc[0]
numbers["S1.31"] = float(adopted["slope"])
print(
    f"adopted definition: qualifying credible sets against log10 effective sample size, "
    f"slope {numbers['S1.31']}, P {adopted['P']:.2e}"
)
sample_size

adopted definition: qualifying credible sets against log10 effective sample size, slope -3.3045, P 1.18e-55


,subset,predictor,n,slope,P,spearman,spearman P
0,all GWAS credible sets,sample size,786140,0.0000,4.205509e-34,0.0973,0.000000e+00
1,all GWAS credible sets,log10 sample size,786140,1.5760,8.880153e-20,0.0973,0.000000e+00
2,all GWAS credible sets,log10 effective sample size,786140,1.6134,7.878120e-34,0.1588,0.000000e+00
3,qualifying credible sets,sample size,520975,0.0000,4.848026e-03,0.0904,0.000000e+00
4,qualifying credible sets,log10 sample size,520975,-1.1713,1.985666e-05,0.0904,0.000000e+00
5,qualifying credible sets,log10 effective sample size,520975,-3.3045,1.178735e-55,0.0489,1.214792e-273


## molQTL credible sets

eQTL counts single-cell eQTL studies alongside bulk eQTL, which is what reproduces 1,402,222.

In [13]:
molqtl = cs.filter(f.col("studyType") != "gwas").cache()
molqtl_studies = si.filter(f.col("studyType") != "gwas").select("studyId", "studyType", "geneId", "biosampleId")
annotated = molqtl.select("studyLocusId", "studyId").join(molqtl_studies, "studyId", "inner").cache()

by_type = annotated.groupBy("studyType").count().toPandas().set_index("studyType")["count"]
print(by_type.sort_values(ascending=False).to_string())

numbers["S1.18"] = int(by_type.get("eqtl", 0) + by_type.get("sceqtl", 0))
numbers["S1.19"] = int(by_type.get("pqtl", 0))
numbers["S1.17"] = annotated.select("geneId").distinct().count()
print({k: numbers[k] for k in ["S1.17", "S1.18", "S1.19"]})
print("tissues or cell types:", annotated.select("biosampleId").distinct().count())

studyType
eqtl      1349478
tuqtl      384852
sqtl       223500
sceqtl      52744
pqtl        33731


{'S1.17': 29342, 'S1.18': 1402222, 'S1.19': 33731}


tissues or cell types: 98


## Replicated credible sets

In [14]:
replicated_gwas = session.spark.read.parquet(paper.derived("replicated_gwas_cs")).count()
replicated_molqtl = session.spark.read.parquet(paper.derived("replicated_molqtl_cs")).count()

numbers["S1.20"] = replicated_gwas
numbers["S1.21"] = round(100 * replicated_gwas / gwas_cs.count(), 1)
numbers["S1.22"] = replicated_molqtl
numbers["S1.23"] = round(100 * replicated_molqtl / molqtl.count(), 1)
print({k: numbers[k] for k in ["S1.20", "S1.21", "S1.22", "S1.23"]})

{'S1.20': 263705, 'S1.21': 33.4, 'S1.22': 1461445, 'S1.23': 71.5}


## Qualified studies and credible sets, and the rare ones among them

In [15]:
qualifying_disease_studies = session.spark.read.parquet(paper.derived("qualifying_gwas_studies"))
qualifying_measurement_studies = session.spark.read.parquet(paper.derived("qualifying_measurement_studies"))
numbers["S1.25"] = qualifying_disease_studies.count()
numbers["S1.24"] = qualifying_measurement_studies.count()

n_qualifying = qualifying_cs.count()

rare = (
    lve.select("studyLocusId", f.col("majorLdPopulationMaf.value").alias("maf"))
    .join(qualifying_cs, "studyLocusId", "inner")
    .filter(f.col("maf") < discovery.MAF_COMMON)
    .count()
)
numbers["S1.26"] = rare
numbers["S1.27"] = round(100 * rare / n_qualifying, 2)
print({k: numbers[k] for k in ["S1.24", "S1.25", "S1.26", "S1.27"]})
print("qualifying credible sets:", f"{n_qualifying:,}")

{'S1.24': 61885, 'S1.25': 15730, 'S1.26': 15311, 'S1.27': 2.94}
qualifying credible sets: 520,975


## Studies excluded before fine-mapping

Around 30% of GWAS Catalog studies were dropped at the study level *before* fine-mapping. This is
the second **precomputed** number: the release index holds only what was ingested, and its
`qualityControls` field records the reasons a study was set aside afterwards — dominated by
"harmonized summary statistics are not available", which is not an exclusion at all, since those
studies are PICS fine-mapped and make up the curated-associations row of Supplementary Table 10.
Excluding that reason leaves about 2%. Both this and the 27.3% above would need the GWAS Catalog
study index from before ingestion (`.../study_index/gwas_catalog` upstream), which is also what
Supplementary Table 10's "Original number of studies before ingestion" column needs.

In [16]:
gcst = si.filter((f.col("studyType") == "gwas") & (f.col("projectId") == "GCST")).cache()
flagged = gcst.filter(f.size(f.coalesce(f.col("qualityControls"), f.array())) > 0)
print(f"GWAS Catalog studies: {gcst.count():,}, carrying a quality-control flag: {flagged.count():,}")
print(f"share flagged: {100 * flagged.count() / gcst.count():.1f}%")

(
    gcst.select(f.explode(f.coalesce(f.col("qualityControls"), f.array())).alias("reason"))
    .groupBy("reason")
    .count()
    .orderBy(f.desc("count"))
    .toPandas()
)

GWAS Catalog studies: 98,223, carrying a quality-control flag: 59,048
share flagged: 60.1%


,reason,count
0,Harmonized summary statistics are not availabl...,56735
1,The number of SNPs in the study is below the e...,2313


## Lead variant-disease pairs seen in more than one study, and their heterogeneity

No surviving code. A pair is the lead variant of a **qualifying disease** credible set together
with one of its study's disease terms — the analysis set the sentence describes. Cochran's Q over
the per-study effect estimates of a pair, `Q = sum(w * b^2) - (sum(w * b))^2 / sum(w)` with
`w = 1 / se^2`, is compared against a chi-square with `k - 1` degrees of freedom, and heterogeneity
is called at `P < 1e-4` as in the manuscript.

Two denominators are possible for the second share, because not every credible set carries a
harmonised effect estimate: the pairs seen in more than one study, or only those of them with at
least two usable estimates. Both are reported; the second is registered, since a pair with one
usable estimate cannot be tested for heterogeneity.

In [17]:
study_records = si.select(
    "studyId", f.concat_ws("|", f.coalesce(f.col("pubmedId"), f.lit("")), f.array_join("cohorts", ",")).alias("record")
)
pairs = (
    lve.filter(f.col("studyStatistics.studyType") == "gwas")
    .select(
        "studyLocusId",
        "variantId",
        "studyId",
        f.explode("diseaseIds").alias("diseaseId"),
        f.col("originalBeta").alias("beta"),
        f.col("originalStandardError").alias("se"),
    )
    .join(study_records, "studyId", "left")
    .cache()
)
qualifying_disease_cs = session.spark.read.parquet(paper.derived("qualifying_credible_sets")).select("studyLocusId")
disease_pairs = pairs.join(qualifying_disease_cs, "studyLocusId", "inner").cache()
qualifying_pairs = pairs.join(qualifying_cs, "studyLocusId", "inner")


def multi_study_share(frame, label):
    """Share of lead variant-disease pairs seen in more than one study, and in more than one publication."""
    per_pair = frame.groupBy("variantId", "diseaseId").agg(
        f.countDistinct("studyId").alias("nStudies"), f.countDistinct("record").alias("nRecords")
    )
    total = per_pair.count()
    return {
        "pair set": label,
        "pairs": total,
        "in >1 study (%)": round(100 * per_pair.filter(f.col("nStudies") > 1).count() / total, 1),
        "in >1 publication or cohort (%)": round(100 * per_pair.filter(f.col("nRecords") > 1).count() / total, 1),
    }


multi_study = pd.DataFrame(
    [
        multi_study_share(disease_pairs, "qualifying disease credible sets"),
        multi_study_share(qualifying_pairs, "qualifying credible sets"),
        multi_study_share(pairs, "all GWAS credible sets"),
    ]
)
multi_study.to_csv(paper.derived("sr1_multi_study_pairs.csv"), index=False)
numbers["S1.29"] = float(multi_study.iloc[0]["in >1 study (%)"])
multi_study

,pair set,pairs,in >1 study (%),in >1 publication or cohort (%)
0,qualifying disease credible sets,60229,16.9,10.5
1,qualifying credible sets,362561,22.9,12.7
2,all GWAS credible sets,572525,20.4,11.1


In [18]:
def cochran(frame, label, multi_study_pairs):
    """Cochran heterogeneity across the effect estimates of each lead variant-disease pair."""
    agg = (
        frame.filter(f.col("beta").isNotNull() & f.col("se").isNotNull() & (f.col("se") > 0))
        .withColumn("w", 1 / (f.col("se") ** 2))
        .groupBy("variantId", "diseaseId")
        .agg(
            f.count("*").alias("k"),
            f.sum("w").alias("sumW"),
            f.sum(f.col("w") * f.col("beta")).alias("sumWB"),
            f.sum(f.col("w") * f.col("beta") ** 2).alias("sumWB2"),
        )
        .filter(f.col("k") > 1)
        .toPandas()
    )
    agg["Q"] = agg["sumWB2"] - agg["sumWB"] ** 2 / agg["sumW"]
    agg["p"] = stats.chi2.sf(agg["Q"].clip(lower=0), agg["k"] - 1)
    significant = int((agg["p"] < 1e-4).sum())
    return {
        "pair set": label,
        "pairs seen in more than one study": multi_study_pairs,
        "pairs with two or more estimates": len(agg),
        "significant at P < 1e-4": significant,
        "of testable pairs (%)": round(100 * significant / len(agg), 1),
        "of multi-study pairs (%)": round(100 * significant / multi_study_pairs, 1),
    }


def multi_study_count(row):
    """Pairs seen in more than one study, from a row of the table above."""
    return int(round(row["pairs"] * row["in >1 study (%)"] / 100))


heterogeneity = pd.DataFrame(
    [
        cochran(disease_pairs, "qualifying disease credible sets", multi_study_count(multi_study.iloc[0])),
        cochran(qualifying_pairs, "qualifying credible sets", multi_study_count(multi_study.iloc[1])),
        cochran(pairs, "all GWAS credible sets", multi_study_count(multi_study.iloc[2])),
    ]
)
heterogeneity.to_csv(paper.derived("sr1_heterogeneity.csv"), index=False)
numbers["S1.30"] = float(heterogeneity.iloc[0]["of testable pairs (%)"])
heterogeneity

,pair set,pairs seen in more than one study,pairs with two or more estimates,significant at P < 1e-4,of testable pairs (%),of multi-study pairs (%)
0,qualifying disease credible sets,10179,2709,281,10.4,2.8
1,qualifying credible sets,83026,21046,5952,28.3,7.2
2,all GWAS credible sets,116795,28055,7686,27.4,6.6


## Write the results

In [19]:
print(paper.save_results("sr01_finemapping_catalogue", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr01_finemapping_catalogue.json


,computed
S1.01,2.024000e+01
S1.11,2.006000e+03
S1.04,1.820000e+01
S1.05,1.370000e+01
S1.03,3.190000e+01
S1.06,5.000000e+01
S1.07,3.150000e+01
S1.08,9.800000e+00
S1.09,8.700000e+00
S1.10,8.270000e+01
